In [ ]:
import numpy as np
import pandas as pd
import scipy
from pathlib import Path
import os
import math

In [ ]:
def find_session_no(session_no_info, mouse_id, date):
    """
    Returns corresponding session number for given mouse id and date.
    
    """
    session_no = session_no_info[(session_no_info['id'] == mouse_id) & 
                             (session_no_info['date'] == date)]['session']
    if len(session_no) == 1:
        session_no = session_no.item()
    else:
        session_no = ''
    return session_no
    
def get_diff(df,col,value):
    """
    Return indices of changes in a binary signal.
    Argument:
        df: Joystick behavior .csv file.
        col: name of the column.
        value: 1 if want to capture 0 to 1 changes. -1 if want to capture 1 to 0 changes.
    """
    dff = df[col].diff()
    idx = df.index[dff==value] - 1
    return idx.values

def butter_low_pass_filter(data, sampling_rate, cutoff_frequency = 20, order = 4):
    """Apply a low-pass Butterworth filter to the data using second-order sections.

    Arguments:
        data: Signal to filter.
        cutoff_frequency: Cutoff frequency to filter in Hz.
        sampling_rate: Sampling rate of the data.
        order: Order of the filter.

    Returns:
        The filtered series.
    """
    nyquist_frequency = sampling_rate / 2 
    cutoff_frequency /= nyquist_frequency

    second_order_sections = scipy.signal.butter(
        order, cutoff_frequency, btype="low", analog=False, output="sos"
    )
    return scipy.signal.sosfiltfilt(second_order_sections, data) 

def calc_tortuosity(start_idx, end_idx, x_array, y_array):
    """Returns tortuosity, trajectory length and distance between start and end point.

    Arguments:
        start_idx: event start index.
        end_idx: event end index.
        x_array: joystick x position timeseries.
        y_array: joystick y position timeseries.
    
    """
    start = start_idx
    end = end_idx
    if np.isnan(start) == False and np.isnan(end) == False:
        total_traj_length = 0
        x_seg = x_array[start:end+1]
        y_seg = y_array[start:end+1]
        for i in range(len(x_seg)-1):
            x1 = x_seg[i]
            x2 = x_seg[i+1]
            y1 = y_seg[i]
            y2 = y_seg[i+1] 
            updated_length = math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            total_traj_length = total_traj_length + updated_length
        distance = math.sqrt((x_seg[-1] - x_seg[0]) ** 2 + (y_seg[-1] - y_seg[0]) ** 2)
        tortuosity = total_traj_length / distance
    else:
        tortuosity = np.nan
        total_traj_length = np.nan
        distance = np.nan
    return tortuosity, total_traj_length, distance


In [ ]:
bh_sampling_rate = 500
bh_sampling_int = 1000 / bh_sampling_rate
center_hold_zone = 2.4
min_rt = 50
velocity_threshold = 30
component_threshold = 10

In [ ]:
session_info_path = '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/session_no_info.csv'
session_no_info = pd.read_csv(session_info_path, converters={'id': str, 'date': str, 'session': str})

paths = ['/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20250108 cohort/preprocess',
        '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20250409 cohort/preprocess',
        '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20250428 cohort/preprocess',
        '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20250714 cohort/preprocess',
        '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20250926 cohort/preprocess',
        '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20251023 cohort/preprocess',
        '/Volumes/sbms/groups/INCR-DuguidLab/sensorimotor control in RTT/FP/20251113 cohort/preprocess']

In [ ]:
component_summary = []
for raw_path in paths:
    base_path = Path(*Path(raw_path).parts[:-1])
    file_names = os.listdir(raw_path)
    folder_names = [name for name in file_names if len(name) == 4]
    folder_paths = [os.path.join(raw_path,name,'bh') for name in folder_names]
    for folder in folder_paths:
        files = os.listdir(folder)
        file_paths = [os.path.join(folder,file) for file in files]
        for file_path in file_paths:
            file_name = os.path.split(file_path)[1]
            mouse_id = file_name[0:4]
            date = file_name[8:16]
            session = find_session_no(session_no_info, mouse_id, date)
            if session != '':
                try:
                    try:
                        bh_df = pd.read_csv(file_path,index_col=False,engine='c')
                    except:
                        bh_df = pd.read_csv(file_path,index_col=False,engine='python')
                        bh_df = bh_df.drop(bh_df.columns[0],axis=1)
                    # Get cue one/off indices.
                    cue_on = get_diff(bh_df,'toneon',1)
                    cue_off = get_diff(bh_df,'toneon',-1)
                    # Remove unpaied cues.
                    if len(cue_off) != len(cue_on):
                        cue_on = cue_on[:-1]
                    # Exclue trials with extra short latency (<=50ms) movements.
                    tone_length = (cue_off - cue_on) * bh_sampling_int
                    tone_on = cue_on[np.where(tone_length > min_rt)[0]]
                    tone_off = cue_off[np.where(tone_length > min_rt)[0]]
                    # Filter x and y signal to calculate speed and radial velocity.
                    filted_x = butter_low_pass_filter(bh_df['x_mm'], bh_sampling_rate)
                    filted_y = butter_low_pass_filter(bh_df['y_mm'], bh_sampling_rate)
                    displacement = np.sqrt(filted_x ** 2+ filted_y ** 2)
                    velocity = raw_v = 1000*np.diff(np.sqrt(filted_x ** 2+filted_y ** 2))/np.diff(bh_df['time_ms'])
                    velocity = np.insert(velocity, 0, 0)
                    speed = 1000*(np.sqrt(np.diff(filted_x)**2+np.diff(filted_y)**2)/np.diff(bh_df['time_ms']))
                    speed = np.insert(speed, 0, 0)
                    # Find time when js crossed centre hold zone.
                    reset_break = np.where(displacement>=center_hold_zone,1,0)
                    for idx, val in enumerate(tone_on):
                        start = tone_on[idx]
                        end = tone_off[idx]
                        trial_v = velocity[start:end]
                        trial_s = speed[start:end]
                        trial_i = bh_df.index[start:end]
                        # MI is the last zero crossing point before velocity reaches 50mm/s.
                        above_threshold_idx = np.where(trial_v > velocity_threshold)[0]
                        if sum(reset_break[start:end])<1 or len(above_threshold_idx) == 0:
                            MI = np.nan
                        else:
                            first_threshold_idx = above_threshold_idx[0]
                            idx_candidates = np.where(trial_v[:first_threshold_idx] <= 0)[0]
                            if len(idx_candidates) == 0:
                                MI = np.nan
                            else:
                                MI = trial_i[idx_candidates[-1]] 
                        # Identify start and end point of movement components using the methods in JOY.
                        if np.isnan(MI) == False:
                            up_cross = np.where((trial_s[:-1] < component_threshold) & (trial_s[1:] >= component_threshold))[0]
                            down_cross = np.where((trial_s[:-1] > component_threshold) & (trial_s[1:] <= component_threshold))[0]
                            if len(down_cross) == 0:
                                up_cross = np.array([idx_candidates[-1]])
                                down_cross = np.array([len(trial_s)-1])
                            if down_cross[-1] <= up_cross[-1]:
                                up_cross = np.delete(up_cross, -1)
                            if down_cross[0] <= up_cross[0]:
                                down_cross = np.delete(down_cross, 0)
                            # Find trial outcome by checking if reward valve opended after cue went off.
                            if sum(bh_df['rvalveopen'].iloc[end:(end + 20)]) >= 1:
                                trial_type = 'correct'
                            else:
                                trial_type = 'failed'
                            for idx in range(len(up_cross)):
                                component_start = up_cross[idx]
                                component_end = down_cross[idx]
                                # Get relevant info of each sub-movement.
                                mean_speed = trial_s[component_start:component_end].mean()
                                max_speed = trial_s[component_start:component_end].max()
                                tort, length, distance = calc_tortuosity(trial_i[component_start], trial_i[component_end], filted_x, filted_y)
                                sp_x = round(filted_x[trial_i[component_start]], 2)
                                sp_y = round(filted_y[trial_i[component_start]], 2)
                                ep_x = round(filted_x[trial_i[component_end]], 2)
                                ep_y = round(filted_y[trial_i[component_end]], 2)  
                                start_time = bh_df['time_ms'].iloc[trial_i[component_start]]
                                end_time = bh_df['time_ms'].iloc[trial_i[component_end]]
                                duration = end_time - start_time
                                component_summary.append({'id': mouse_id, 'session': session, 'trial_type': trial_type, 
                                                          'MI': bh_df['time_ms'].iloc[MI],'component_number': idx+1, 
                                                          'start': start_time, 'end':end_time, 'duration': duration, 
                                                          'mean_speed': mean_speed, 'max_speed': max_speed, 'tortuosity': tort,
                                                     'traj_length': length, 'distance': distance, 'sp_x': sp_x, 'sp_y': sp_y,
                                                          'ep_x': ep_x, 'ep_y': ep_y})   
                    print(f'{file_name} analysis finished')
                except Exception as e:
                    print(f'error in {file_name},{e}')
component_summary = pd.DataFrame(component_summary)